In [1]:
# Import libraries

import os
import zipfile
import random

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    # Raw zip file
    "brats_zip": os.path.join(BASE_DIR, "data", "raw", "brats20-nifti.zip"),

    # Extracted dataset folder
    "brats_dir": os.path.join(BASE_DIR, "data", "brats2020"),

    # Output folders
    "figures": os.path.join(BASE_DIR, "results", "figures"),
    "metrics": os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:12s} -> {path}")

In [ ]:
# Extract Dataset
# Run this once to extract the zip file into its target folder

def extract_zip(zip_path, target_dir, dataset_name):
    if not os.path.exists(zip_path):
        print(f"ERROR: zip not found -> {zip_path}")
        return False

    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"{dataset_name}: already extracted, skipping.")
        return True

    print(f"Extracting {dataset_name}...")
    os.makedirs(target_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target_dir)

    print(f"{dataset_name} extracted to: {target_dir}")
    return True


extract_zip(PATHS["brats_zip"], PATHS["brats_dir"], "BraTS2020")

In [ ]:
# Verify Folder Structure
# Walk down into the extracted folder to find the actual patient directories

brats_dir = PATHS["brats_dir"]

top_level = os.listdir(brats_dir)
print(f"Top-level contents: {top_level}")

# Go one level deeper
level2_path = os.path.join(brats_dir, top_level[0])
level2 = os.listdir(level2_path)
print(f"\nInside '{top_level[0]}': {level2}")

# Go one more level deeper (expecting the actual patient folders here)
level3_path = os.path.join(level2_path, level2[0])
if os.path.isdir(level3_path):
    level3 = os.listdir(level3_path)
    print(f"\nInside '{level2[0]}': {len(level3)} items")
    print("First 5 items:")
    for item in level3[:5]:
        print(f"  {item}")

In [ ]:
# Inspect a single patient folder to confirm the 5 expected NIfTI files

training_dir = os.path.join(brats_dir, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

# Filter out the CSV files, keep only patient folders
patient_folders = sorted([
    f for f in os.listdir(training_dir)
    if os.path.isdir(os.path.join(training_dir, f))
])

print(f"Total patient folders: {len(patient_folders)}")

sample_patient = patient_folders[0]
sample_path = os.path.join(training_dir, sample_patient)

print(f"\nFiles inside '{sample_patient}':")
for f in os.listdir(sample_path):
    file_path = os.path.join(sample_path, f)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
# Load a single NIfTI volume and inspect its properties

sample_patient = "BraTS20_Training_001"
patient_dir = os.path.join(training_dir, sample_patient)

flair_path = os.path.join(patient_dir, f"{sample_patient}_flair.nii")
seg_path = os.path.join(patient_dir, f"{sample_patient}_seg.nii")

flair_img = nib.load(flair_path)
seg_img = nib.load(seg_path)

flair_data = flair_img.get_fdata()
seg_data = seg_img.get_fdata()

print(f"FLAIR volume shape: {flair_data.shape}")
print(f"FLAIR data type: {flair_data.dtype}")
print(f"FLAIR value range: [{flair_data.min():.1f}, {flair_data.max():.1f}]")
print()
print(f"Segmentation mask shape: {seg_data.shape}")
print(f"Unique labels in mask: {np.unique(seg_data)}")
print()
print("Affine matrix (voxel-to-world coordinate mapping):")
print(flair_img.affine)

In [ ]:
# Visualize a mid-depth slice with the tumor mask overlaid

mid_slice = flair_data.shape[2] // 2  # roughly the middle of the volume

flair_slice = flair_data[:, :, mid_slice]
seg_slice = seg_data[:, :, mid_slice]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(flair_slice.T, cmap="gray", origin="lower")
axes[0].set_title(f"FLAIR — slice {mid_slice}")
axes[0].axis("off")

axes[1].imshow(flair_slice.T, cmap="gray", origin="lower")
masked_seg = np.ma.masked_where(seg_slice.T == 0, seg_slice.T)
axes[1].imshow(masked_seg, cmap="autumn", alpha=0.5, origin="lower")
axes[1].set_title(f"FLAIR + Tumor Mask — slice {mid_slice}")
axes[1].axis("off")

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "sample_slices.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Load and compare all 4 modalities for the same patient and slice

modalities = ["t1", "t1ce", "t2", "flair"]
volumes = {}

for mod in modalities:
    mod_path = os.path.join(patient_dir, f"{sample_patient}_{mod}.nii")
    volumes[mod] = nib.load(mod_path).get_fdata()

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, mod in zip(axes, modalities):
    ax.imshow(volumes[mod][:, :, mid_slice].T, cmap="gray", origin="lower")
    ax.set_title(mod.upper())
    ax.axis("off")

fig.suptitle(f"{sample_patient} — All Modalities, slice {mid_slice}", fontsize=13)
plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "modality_comparison.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Compute tumor volume statistics across a sample of patients
# (using a subset for speed; full dataset stats can follow the same pattern)

sample_size = 50
sample_patients = random.sample(patient_folders, sample_size)

tumor_voxel_ratios = []
tumor_voxel_counts = []

for pid in sample_patients:
    seg_file = os.path.join(training_dir, pid, f"{pid}_seg.nii")
    seg = nib.load(seg_file).get_fdata()

    tumor_voxels = np.sum(seg > 0)
    total_voxels = seg.size

    tumor_voxel_counts.append(tumor_voxels)
    tumor_voxel_ratios.append(tumor_voxels / total_voxels * 100)

print(f"Tumor volume statistics (n={sample_size} patients)")
print("-" * 45)
print(f"Tumor voxel count  min: {min(tumor_voxel_counts):,}")
print(f"Tumor voxel count  max: {max(tumor_voxel_counts):,}")
print(f"Tumor voxel count  avg: {np.mean(tumor_voxel_counts):,.0f}")
print()
print(f"Tumor % of volume  min: {min(tumor_voxel_ratios):.2f}%")
print(f"Tumor % of volume  max: {max(tumor_voxel_ratios):.2f}%")
print(f"Tumor % of volume  avg: {np.mean(tumor_voxel_ratios):.2f}%")
print(f"Tumor % of volume  med: {np.median(tumor_voxel_ratios):.2f}%")

In [ ]:
# Visualize the distribution of tumor sizes across the sampled patients

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(tumor_voxel_ratios, bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(np.mean(tumor_voxel_ratios), color="coral", linestyle="--", linewidth=1.5, label="mean")
axes[0].set_title("Tumor Size Distribution (% of brain volume)")
axes[0].set_xlabel("Tumor % of total volume")
axes[0].set_ylabel("Number of patients")
axes[0].legend()

axes[1].boxplot(tumor_voxel_ratios, vert=True)
axes[1].set_title("Tumor Size Spread")
axes[1].set_ylabel("Tumor % of total volume")

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "tumor_size_distribution.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Summary

print("NOTEBOOK 01 COMPLETE")
print("=" * 50)
print(f"Dataset: BraTS2020, {len(patient_folders)} patients")
print(f"Volume shape: {flair_data.shape} (4 modalities: T1, T1ce, T2, FLAIR)")
print(f"Segmentation labels: {[int(l) for l in np.unique(seg_data)]} (0=background, 1=NCR/NET, 2=ED, 4=ET)")
print()
print(f"Tumor size (n={sample_size} sampled patients):")
print(f"  avg: {np.mean(tumor_voxel_ratios):.2f}% of total volume")
print(f"  range: {min(tumor_voxel_ratios):.2f}% - {max(tumor_voxel_ratios):.2f}%")
print(f"  -> severe class imbalance, will need Dice + Focal loss and foreground-biased patch sampling")
print()
print("Figures saved:")
for fname in ["sample_slices.png", "modality_comparison.png", "tumor_size_distribution.png"]:
    path = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Next -> 02_data_preparation.ipynb")
print("  - Define patch extraction strategy (foreground-biased sampling)")
print("  - Train/val/test split")
print("  - Set up MONAI dataset/dataloader pipeline")